# Integrity Tool — Phase 1 (ESA CCI-LC 300 m, 2000–2015)

Global country-level agricultural integrity in Earth Engine. For each country and year:
- **`gritM`** — fraction of that country's agricultural cells whose ~1 km ring holds ≥ `THRESHOLD_PCT` % natural land.
- **`agArea_ha`** — agricultural area (ha).

Validated to match the original arcpy pipeline closely.

---
## How to run

**One-time (S0):** prepare & ingest the GADM boundary asset. Never again after that.

**Every session, top to bottom:**
1. **§1** Auth → **§2** Config → **§3–§5** Definitions
2. **§6** Country→tile assignment  *(the single assignment that governs everything)*
3. **§7** *(optional)* single-tile test
4. **§8** Export all tiles × years  → CSVs in Drive
5. **§9** *(after exports finish)* combine → one wide CSV
6. **§10** verify

> After a runtime restart, use **Runtime → Run all** — the S0 cell is commented so it's skipped.

---
## Architecture (why it's built this way)

**One assignment governs both export and combine.** §6 computes, for each country, the tile whose
box overlaps it most (in geopandas, once). §8 exports *only* each tile's assigned countries, so every
country is computed in **exactly one** tile. §9 is therefore a plain concatenation — no dedup, no
possibility of a country being double-counted or silently dropped. (This replaces the earlier design
where server-side centroid subsetting and area-based combining could disagree.)

## Key gotchas (learned the hard way — don't re-litigate)
- **Asset starts in 2000.** No 1992–1999 images in this asset → `YEARS = 2000–2015`.
- **No map projection is set.** Mollweide / EPSG:6933 failed to parse or threw edge-transform errors.
  `pixelArea()` gives true areas; the ring uses **pixel** kernels — so no CRS anywhere.
- **Tiling is mandatory.** A global `reduceNeighborhood` fails at the ±180/±90 raster edge; per-tile
  processing keeps every computation away from those corners.
- **Fresh output folder per run.** `DRIVE_FOLDER` is versioned so a re-export can't mix with stale CSVs
  from an earlier tile design.
- **Free-tier quota** throttles to "restricted mode"; batch tasks queue (READY) and drain slowly. Phase 2
  (10 m) will need billing or a quota bump.

## Known limitation
Countries larger than their assigned tile, or straddling a seam by more than `BUFFER_KM`
(Russia, Canada), can under-count: only the portion inside the assigned tile's buffered region is
measured. Eyeball those in §10. Full fix (sum a country across all tiles it touches) is a future
enhancement, deferred because the validated run's giant-country values were acceptable.

## Follow-on (not this notebook)
- **Phase 2:** 10 m land cover (WorldCover 2020/21 or Dynamic World ≥2015); different classes, recent
  snapshots, heavier compute; reuses this tiling skeleton.
- **1990s:** splice 1992–1999 from the separate ESA-CCI asset; check the ESA-CCI→C3S discontinuity at 2000.


## S0 · One-time GADM setup  ⚠️ RUN ONCE

EE needs boundaries as an **ingested table asset**. Global GADM has one country above EE's
1,000,000-vertex limit, so simplify first, then ingest via the Code Editor.

1. Uncomment & run the cell once (reads raw GADM from Drive → writes simplified).
2. Code Editor → Assets → New → Shape files: upload the simplified `.shp` + `.shx`/`.dbf`/`.prj`
   (WGS84). Name it to match `GADM_ASSET` in §2.


In [ ]:
# ⚠️ RUN ONCE. Commented so "Run all" skips it.
# from google.colab import drive; drive.mount("/content/drive")
# import geopandas as gpd
# RAW = "/content/drive/MyDrive/Integrity/GADM_GID_0.shp"
# OUT = "/content/drive/MyDrive/Integrity/GADM_GID_0_simplified.shp"
# g = gpd.read_file(RAW)
# assert str(g.crs).upper().endswith("4326"), "need EPSG:4326 so 0.01 deg ~ 1 km"
# g["geometry"] = g.geometry.simplify(0.01, preserve_topology=True)   # raise tol if any feature > 1e6 verts
# def _vc(x):
#     if x is None: return 0
#     gs = x.geoms if x.geom_type.startswith("Multi") else [x]
#     return sum(len(p.exterior.coords)+sum(len(r.coords) for r in p.interiors) for p in gs)
# print("max vertices (< 1,000,000):", g.geometry.apply(_vc).max())
# g.to_file(OUT); print("saved", OUT)


## 1 · Setup & authentication

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project="ee-fremier")   # <-- your EE Cloud project ID
print("EE initialised")


## 2 · Configuration

`TILES` is the single source of truth (used by §6 assignment, §8 export, and — via §6 — §9 combine).
Boxes may overlap; the area-based assignment resolves that. Together they cover all inhabited land.
`DRIVE_FOLDER` is **versioned** so this run's CSVs never mix with an earlier design's.


In [ ]:
# ---- Boundaries -----------------------------------------------------------
GADM_ASSET = "projects/ee-fremier/assets/GADM_GID_0_simplified"   # ingested EE asset (§8 reduceRegions)
GADM_SHP   = "/content/drive/MyDrive/Integrity/GADM_GID_0_simplified.shp"  # local copy (§6 assignment)
ID_FIELD   = "GID_0"

# ---- Land cover -----------------------------------------------------------
LC_COLLECTION = "projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS"
LC_BAND       = "b1"

# ---- Analysis parameters --------------------------------------------------
YEARS         = list(range(2000, 2016))   # asset has 2000..2015 only
SCALE         = 300           # metres (native LCCS resolution)
KERNEL_OUTER  = 3             # pixels (~900 m ring outer)
KERNEL_INNER  = 1             # pixels (excludes focal + rook neighbours)
THRESHOLD_PCT = 20            # integrity threshold, percent
BUFFER_KM     = 25            # processing overlap so edge/border rings have data
DRIVE_FOLDER  = "EE_Integrity_v2"   # <-- versioned; fresh folder for this run

# ---- Tiles: [west, south, east, north] degrees ----------------------------
TILES = {
    "N_America":       [-170,   5,  -50,  75],
    "S_America":       [ -95, -60,  -30,  15],
    "Europe":          [ -25,  34,   45,  75],
    "Africa":          [ -20, -40,   55,  38],
    "Asia":            [  45,   5,  150,  78],
    "SE_Asia_Oceania": [  90, -50,  180,  12],
    "Pacific_East":    [-180, -30,  -95,  30],   # Samoa, Cook, Niue, Tonga, Fr. Polynesia
    "Arctic":          [-180,  60,  180,  82],   # Greenland, Iceland, Svalbard (<82N, off pole)
    "Indian_Is":       [  40, -35,   95,  10],   # Maldives, Mauritius, Reunion, Seychelles
    "Cape_Verde":      [ -30,  10,  -20,  20],   # Cape Verde
    # Antarctica intentionally excluded — polar projection fails export;
    # no sovereign countries / no land-cover-integrity signal (permanent ice).
}
print(len(TILES), "tiles:", list(TILES.keys()))


## 3 · Reclass + agricultural mask

Fractional natural integrity (0..1): crop/urban = 0, mosaic-30 = 0.25, mosaic-40 = 0.75,
natural veg & bare = 1. Water (210) / snow-ice (220) absent → **masked**, matching arcpy
`Reclassify(..., "NODATA")`. Ag mask = classes 10–41 (any ag, incl. both mosaics).


In [ ]:
NATURAL = [50,60,61,62,70,71,72,80,81,82,90,100,110,120,121,122,
           130,140,150,151,152,153,160,170,180]
BARE, CROP, URBAN = [200,201,202], [10,11,12,20], [190]

INTEGRITY = {c:0.0 for c in CROP+URBAN}
INTEGRITY[30] = 0.25; INTEGRITY[40] = 0.75
for c in NATURAL+BARE: INTEGRITY[c] = 1.0
_FROM = list(INTEGRITY.keys()); _TO = [INTEGRITY[k] for k in _FROM]

def reclass_integrity(lc):
    return lc.remap(_FROM, _TO).rename("natgrit").toFloat()   # water/ice masked (not in map)

def ag_mask(lc):
    return lc.gte(10).And(lc.lte(41)).rename("ag")


## 4 · Annulus focal mean (pixel kernels)

Mean natural integrity in a ~1 km ring = (outer circle − inner circle) for sum and count. EE's
neighbourhood reducers skip masked pixels → reproduces arcpy `FocalStatistics(..., "DATA")`.

> Inner radius 1 excludes the focal cell **and** its 4 rook neighbours (not the focal cell alone) —
> a minor, accepted difference from `NbrAnnulus(0.5, 3)`; confirmed "close enough" vs. arcpy.


In [ ]:
def annulus_mean(natgrit):
    outer = ee.Kernel.circle(radius=KERNEL_OUTER, units="pixels", normalize=False)
    inner = ee.Kernel.circle(radius=KERNEL_INNER, units="pixels", normalize=False)
    so = natgrit.reduceNeighborhood(ee.Reducer.sum(),   outer)
    si = natgrit.reduceNeighborhood(ee.Reducer.sum(),   inner)
    co = natgrit.reduceNeighborhood(ee.Reducer.count(), outer)
    ci = natgrit.reduceNeighborhood(ee.Reducer.count(), inner)
    ring_sum = so.subtract(si)
    ring_cnt = co.subtract(ci)
    ring_cnt = ring_cnt.where(ring_cnt.lte(0), ee.Image(0).mask(ee.Image(0)))
    return ring_sum.divide(ring_cnt).rename("focal")


## 5 · Per-tile processing

`process_tile(box, gid_list, year)` processes a tile on its buffered region and reduces over only
the countries assigned to that tile (`gid_list`, from §6). Area via `pixelArea()` (true area).


In [ ]:
countries = ee.FeatureCollection(GADM_ASSET)
lc_ic     = ee.ImageCollection(LC_COLLECTION)

def year_image(year, region):
    year = ee.Number(year)
    img = lc_ic.filterDate(ee.Date.fromYMD(year,1,1), ee.Date.fromYMD(year,12,31)).first()
    return ee.Image(img).select([LC_BAND]).rename("lccs").clip(region)

def processing_region(box):
    return ee.Geometry.Rectangle(box, proj="EPSG:4326", geodesic=False).buffer(BUFFER_KM * 1000)

def process_tile(box, gid_list, year):
    year   = ee.Number(year)
    region = processing_region(box)
    subset = countries.filter(ee.Filter.inList(ID_FIELD, gid_list))   # only this tile's countries

    lc        = year_image(year, region)
    natgrit   = reclass_integrity(lc)
    ag        = ag_mask(lc)
    focal_pct = annulus_mean(natgrit).multiply(100).toInt()
    con_ag    = focal_pct.gte(THRESHOLD_PCT).updateMask(ag).rename("grit")
    ag_area   = ee.Image.pixelArea().updateMask(ag).divide(10000).rename("agarea")
    stats     = con_ag.addBands(ag_area)

    reducer = ee.Reducer.mean().combine(ee.Reducer.sum(), sharedInputs=True)
    fc = stats.reduceRegions(collection=subset, reducer=reducer, scale=SCALE)

    def tidy(f):
        return ee.Feature(None, {
            ID_FIELD:    f.get(ID_FIELD),
            "year":      year,
            "gritM":     f.get("grit_mean"),
            "agArea_ha": f.get("agarea_sum"),
        })
    return fc.map(tidy)


## 6 · Country → tile assignment  (the one assignment that governs everything)

Each country is assigned to the tile whose box overlaps it most. This dict drives **both** the
export (§8, which processes only a tile's assigned countries) and, implicitly, the combine (§9,
a plain concat). One country → one tile → one row per year. No dupes, no drops possible.


In [ ]:
import geopandas as gpd
from shapely.geometry import box as shp_box
from google.colab import drive
drive.mount("/content/drive")

gdf = gpd.read_file(GADM_SHP).to_crs("EPSG:4326")
gdf["geometry"] = gdf.geometry.buffer(0)          # repair invalid polygons (simplification artifacts)

tile_boxes = {n: shp_box(w, s, e, nn) for n, (w, s, e, nn) in TILES.items()}
def best_tile(geom):
    if geom is None: return None
    best, ba = None, -1.0
    for name, tb in tile_boxes.items():
        try: a = geom.intersection(tb).area
        except Exception: a = 0.0
        if a > ba: best, ba = name, a
    return best

gdf["best"] = gdf.geometry.apply(best_tile)
assign    = dict(zip(gdf[ID_FIELD], gdf["best"]))                 # GID_0 -> tile
tile_gids = {t: sorted(gdf.loc[gdf["best"] == t, ID_FIELD]) for t in TILES}   # tile -> [GID_0]

n_assigned = sum(len(v) for v in tile_gids.values())
print("countries assigned:", n_assigned, "of", len(gdf))
print("unassigned (null geom):", gdf[ID_FIELD][gdf["best"].isna()].tolist() or "none")
for t, gids in tile_gids.items():
    print(f"  {t:16s} {len(gids)}")


## 8 · Export all tiles × years

One batch export per (tile, year) into the versioned Drive folder. Under quota restriction tasks
sit in READY and drain slowly — expected; don't re-run (it just queues duplicates).


In [ ]:
tasks = []
for tname, box in TILES.items():
    gids = tile_gids.get(tname, [])
    if not gids:
        print("skip (no countries):", tname); continue
    for yr in YEARS:
        fc = process_tile(box, gids, yr)
        t = ee.batch.Export.table.toDrive(
            collection=fc, description=f"integ_{tname}_{yr}",
            folder=DRIVE_FOLDER, fileNamePrefix=f"integ_{tname}_{yr}",
            fileFormat="CSV", selectors=[ID_FIELD, "year", "gritM", "agArea_ha"],
        )
        t.start(); tasks.append(t)
    print("queued:", tname)
print(f"\n{len(tasks)} export tasks started -> Drive/{DRIVE_FOLDER}")


In [ ]:
# Status monitor — re-run until all COMPLETED.
from collections import Counter
print(Counter(t.status()["state"] for t in tasks))


## 9 · Combine → one wide CSV  (after all exports COMPLETED)

Plain concat — each country appears exactly once (single assignment in §6), so no dedup is needed.
An assertion guards that invariant. Output columns match arcpy: `GritM_<year>`, `AgArea_<year>`.


In [ ]:
import pandas as pd, glob, os
d = f"/content/drive/MyDrive/{DRIVE_FOLDER}"

# Load every exported CSV. Sharded tile exports repeat each country across shards,
# and summed areas carry last-bit float drift, so the raw load has ~2x rows.
# Tag each row with its tile, then collapse to one row per country x year x tile.
frames = []
for f in sorted(glob.glob(os.path.join(d, "integ_*.csv"))):
    stem = os.path.basename(f)[len("integ_"):-len(".csv")]   # e.g. Asia_2000
    df = pd.read_csv(f)
    df["tile"] = stem.rsplit("_", 1)[0]
    frames.append(df)

long_df = (pd.concat(frames, ignore_index=True)
             .groupby([ID_FIELD, "year", "tile"], as_index=False)
             .agg(gritM=("gritM", "mean"), agArea_ha=("agArea_ha", "mean")))

# After the collapse, any remaining duplicate is a TRUE cross-tile straddler
# (same country under two different tile labels), not shard noise.
dup = long_df.groupby([ID_FIELD, "year"]).size().max()
assert dup == 1, f"expected 1 row per country-year, found {dup} — real cross-tile straddler?"
print("rows:", len(long_df), "| countries:", long_df[ID_FIELD].nunique())

# Save the clean long series (canonical, for time-series/analysis)
long_df.sort_values([ID_FIELD, "year"]).to_csv(
    os.path.join(d, "integrity_clean_2000_2015.csv"), index=False)

# ...and the wide series (one row per country, for joins/mapping).
grit = long_df.pivot(index=ID_FIELD, columns="year", values="gritM").add_prefix("GritM_")
area = long_df.pivot(index=ID_FIELD, columns="year", values="agArea_ha").add_prefix("AgArea_")
wide = grit.join(area).reset_index()
wide.to_csv(os.path.join(d, "IntegritySeries_2000_2015.csv"), index=False)
print("saved long + wide CSVs to", d)
wide.head()


## 10 · Verify

In [ ]:
check = ['WSM','COK','NIU','TON','PYF','ASM','PCN','GRL','ISL','CPV','MDV',
         'MUS','REU','SYC','RUS','CAN','NOR','FIN','PHL']
have = set(wide[ID_FIELD])
for g in check:
    print(f"{g}: {'present' if g in have else 'MISSING'}")
print("\ntotal countries:", len(wide))

print("\nsanity — AgArea_2015 (ha) for giant/large countries:")
print(wide[wide[ID_FIELD].isin(['RUS','CAN','USA','CHN','IND'])]
      [[ID_FIELD, "AgArea_2015"]].to_string(index=False))


## Notes

- **Changing `TILES`** → re-run §6 (reassigns countries) then re-export §8. §6 and §8/§9 must
  always use the same `TILES`.
- **`DRIVE_FOLDER` is versioned.** To re-run cleanly, bump the version (e.g. `_v3`) so no stale CSVs
  mix in.
- **§9 collapses shard / float-noise duplicates** (groupby-mean on `[GID_0, year, tile]`) before the
  assertion, so the assertion now fires only on a *true* cross-tile straddler — a country appearing
  under two different tile labels. §9 writes both the long (`integrity_clean_*`) and wide
  (`IntegritySeries_*`) CSVs.
- **Antarctica excluded** — polar projection fails export; no sovereign countries / no
  land-cover-integrity signal (permanent ice).
- **`BUFFER_KM = 25`** only affects the focal computation, not attribution; larger is safe.
- **Giant-country undercount** (see header) — Russia/Canada may read slightly low; §10 prints them.
